# [Chapter 13 — Sexual Transmission, §13.6] Assortative vs Proportional Mixing: Effect on $\mathcal{R}_0$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 13 — Sexual Transmission
**Considerations developed:** 7 (mixing balance), 12 (basic vs effective $R$ misapplication)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_13_sexual_transmission/02_assortative_mixing_R0.ipynb)


## What this notebook does

For a population stratified by sexual activity, mixing patterns interpolate between two extremes: *proportional* (random) mixing where partners are drawn in proportion to their activity, and *fully assortative* mixing where individuals only partner within their own activity class. The notebook computes the dominant eigenvalue of the next-generation matrix as a function of an assortativity parameter $\epsilon \in [0, 1]$ and shows that increasing assortativity raises $\mathcal{R}_0$ — a phenomenon often missed by analysts who silently assume proportional mixing.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_13
from shared.verification import assert_within_tolerance
set_seed_chapter_13()
book_style()


## Two-activity-class population

Group L (low activity, 80% of population, $c_L = 1$ partnership/year) and group H (high activity, 20% of population, $c_H = 5$ partnerships/year). The mixing matrix interpolates between proportional ($\epsilon = 0$) and fully assortative ($\epsilon = 1$):
$$M_{ij}(\epsilon) = \epsilon\,\delta_{ij} + (1-\epsilon)\,\frac{c_j N_j}{\sum_k c_k N_k}.$$

In [ ]:
NL, NH = 0.8, 0.2
cL, cH = 1.0, 5.0
beta = 0.10  # per-partnership transmission probability
tau = 1.0    # year of infectious duration

def mixing(eps):
    denom = cL*NL + cH*NH
    prop = np.array([[cL*NL, cH*NH],[cL*NL, cH*NH]])/denom
    return eps*np.eye(2) + (1-eps)*prop

def R0_NGM(eps):
    M = mixing(eps)
    K = np.array([[beta*cL*M[0,0]*tau, beta*cL*M[0,1]*tau],
                  [beta*cH*M[1,0]*tau, beta*cH*M[1,1]*tau]])
    return float(np.max(np.real(np.linalg.eigvals(K))))

eps_grid = np.linspace(0.0, 1.0, 51)
R0_curve = np.array([R0_NGM(e) for e in eps_grid])

print(f'R_0 at fully proportional mixing (eps=0): {R0_curve[0]:.3f}')
print(f'R_0 at fully assortative mixing  (eps=1): {R0_curve[-1]:.3f}')
print(f'Inflation factor: {R0_curve[-1]/R0_curve[0]:.3f}x')


## $\mathcal{R}_0(\epsilon)$ curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(eps_grid, R0_curve, color=BOOK_COLORS['primary'], lw=2)
ax.axhline(1.0, ls='--', color='gray', alpha=0.6, label='threshold')
ax.set_xlabel('assortativity $\\epsilon$')
ax.set_ylabel('R_0 (dominant eigenvalue of NGM)')
ax.set_title('Assortative mixing inflates R_0')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verify monotonicity
assert np.all(np.diff(R0_curve) >= -1e-10), 'R_0 should be monotone non-decreasing in epsilon'
print('Verified: R_0 is monotone non-decreasing in assortativity.')


### Why this matters operationally (Consideration~12)

If a modeler estimates $\beta$ by fitting an early-outbreak slope to $\mathcal{R}_0$, the inferred $\beta$ depends on the assumed $\epsilon$. Using $\epsilon = 0$ when the true mixing is partially assortative *overestimates* $\beta$, which then propagates to *underestimating* required intervention coverage. Reporting $\mathcal{R}_0$ without specifying the mixing assumption is a Consideration-12 violation.

## What's next

- Chapter 14 transfers this NGM machinery to vector-host transmission cycles.
- Chapter 19 (HIV) and Chapter 20 (cross-pathogen synthesis) revisit assortative mixing in real datasets.
- The same $\epsilon$ parameter governs the mixing matrix used for targeted-intervention sensitivity analysis in Chapter 10.